01_business_understanding.ipynb
02_data_quality_assessment.ipynb
03_etl_pipeline.ipynb
04_dimensional_data_modeling.ipynb
05_sql_analytics.ipynb
06_exploratory_data_analysis.ipynb
07_kpi_framework.ipynb
08_root_cause_analysis.ipynb
09_dashboard_development.ipynb
10_business_recommendations.ipynb

# Phase 3 — ETL Workflow

## What is ETL?

**ETL = Extract → Transform → Load**

It is the process of taking raw data, preparing it for analysis, and storing the cleaned output for downstream use.

| Stage         | In This Project                                             |
| ------------- | ----------------------------------------------------------- |
| **Extract**   | Load the raw retail transaction data                        |
| **Transform** | Clean and standardize the dataset based on Phase 2 findings |
| **Load**      | Save the cleaned dataset for data modeling and analysis     |

### Workflow

```text
Raw Data
   ↓
Extract
   ↓
Transform
   ↓
Validate
   ↓
Load
   ↓
Clean Analytical Data
```

## Objective

Transform the raw retail transaction data into a clean, analysis-ready dataset while preserving valid records and the original **transaction-line-item grain**.

Issues identified during Data Quality Assessment

✓ Unnamed column

✓ Date stored as object

✓ Column naming inconsistent

✓ Dataset unsorted


The ETL pipeline addresses each issue.

The cleaned dataset will serve as the foundation for the **Star Schema Design** and downstream analytics.


## **1. Extract**

The raw retail transaction dataset is loaded from the project's raw data directory.

This dataset serves as the input for the transformation steps identified during the Data Quality Assessment.

In [1]:
import pandas as pd

In [2]:
# Load raw dataset
df = pd.read_csv("../data/raw/scanner_data.csv")

# Check the extracted dataset
df.shape

(131706, 8)

In [3]:
df.columns

Index(['Unnamed: 0', 'Date', 'Customer_ID', 'Transaction_ID', 'SKU_Category',
       'SKU', 'Quantity', 'Sales_Amount'],
      dtype='object')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131706 entries, 0 to 131705
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Unnamed: 0      131706 non-null  int64  
 1   Date            131706 non-null  object 
 2   Customer_ID     131706 non-null  int64  
 3   Transaction_ID  131706 non-null  int64  
 4   SKU_Category    131706 non-null  object 
 5   SKU             131706 non-null  object 
 6   Quantity        131706 non-null  float64
 7   Sales_Amount    131706 non-null  float64
dtypes: float64(2), int64(3), object(3)
memory usage: 8.0+ MB


## **2. Transform**

### 2.1 Remove Unnecessary Source Index

The `Unnamed: 0` column was identified during the Data Quality Assessment as a source-generated index column. It contains sequential values that duplicate the DataFrame index and does not provide meaningful business information.

Since it is not required for analysis or data modeling, it will be removed from the analytical dataset.


In [5]:
df =df.drop(columns=['Unnamed: 0'])

In [6]:
df.columns

Index(['Date', 'Customer_ID', 'Transaction_ID', 'SKU_Category', 'SKU',
       'Quantity', 'Sales_Amount'],
      dtype='object')

In [7]:
assert 'Unnamed: 0' not in df.columns

**Finding**

The unnecessary Unnamed: 0 column has been removed. The remaining columns represent the actual retail transaction data required for downstream analysis and modeling.

### 2.2 Convert Date to Datetime

The `Date` column was identified during the Data Quality Assessment as an `object` data type. It will be converted to `datetime` to support time-based analysis, date filtering, and monthly or daily aggregations.


In [8]:
df['Date'] = pd.to_datetime(
    df['Date'], format='%d/%m/%Y'   
)

In [9]:
df['Date'].dtype

# Note:
# dtype('<M8[ns]') is NumPy's internal representation of datetime64[ns].
# M8 → datetime data type
# ns → nanosecond precision
# So dtype('<M8[ns]') and datetime64[ns] mean the same thing.

dtype('<M8[ns]')

In [10]:
df[['Date']].head()

,Date
0,2016-01-02
1,2016-01-02
2,2016-01-02
3,2016-01-02
4,2016-01-02


In [11]:
assert pd.api.types.is_datetime64_any_dtype(df['Date'])

**Finding**

The Date column has been successfully converted from object to datetime, making it suitable for time-based analysis and downstream data modeling.

### 2.3 Standardize Column Naming Convention


Standardize all column names using a consistent `snake_case` naming convention to improve readability, maintainability, and compatibility across Python, SQL, and downstream analytical workflows.

In [12]:
df.columns

Index(['Date', 'Customer_ID', 'Transaction_ID', 'SKU_Category', 'SKU',
       'Quantity', 'Sales_Amount'],
      dtype='object')

In [13]:
df = df.rename(columns = {'Date': 'date', 'Customer_ID': 'customer_id', 'Transaction_ID': 'transaction_id', 'SKU_Category': 'sku_category', 'SKU': 'sku', 'Quantity': 'quantity', 'Sales_Amount': 'sales_amount'})

In [14]:
df.columns

Index(['date', 'customer_id', 'transaction_id', 'sku_category', 'sku',
       'quantity', 'sales_amount'],
      dtype='object')

In [15]:
assert list(df.columns) == ['date', 'customer_id', 'transaction_id', 'sku_category', 'sku', 'quantity', 'sales_amount']

The cleaned dataset adopts a consistent snake_case naming convention for all columns. Standardized column names improve readability, maintainability, and compatibility across Python, SQL, and downstream analytical tools while preserving the original business meaning of each field.

**Finding:**
 All business columns were successfully renamed using a consistent snake_case convention without altering the underlying data.

## 2.4 Sort Dataset

Sort the cleaned dataset by `date` and `transaction_id` to establish a consistent record order, improving reproducibility, validation, and manual inspection without altering the underlying data.

In [16]:
df = df.sort_values(
    ['date', 'transaction_id']
).reset_index(drop=True)

In [17]:
df.head()

,date,customer_id,transaction_id,sku_category,sku,quantity,sales_amount
0,2016-01-02,2547,1,X52,0EM7L,1.0,3.13
1,2016-01-02,822,2,2ML,68BRQ,1.0,5.46
2,2016-01-02,3686,3,0H2,CZUZX,1.0,6.35
3,2016-01-02,3719,4,0H2,549KK,1.0,5.59
4,2016-01-02,9200,5,0H2,K8EHH,1.0,6.88


The cleaned dataset was sorted by date and transaction_id to produce a consistent and chronological record order. Sorting improves readability, simplifies validation and debugging, and ensures reproducible outputs without altering the underlying data.

**Finding:** The dataset was successfully ordered chronologically while preserving all records and the original transaction-line-item grain.

# **3. Validate Transformations**

After applying the transformations, the dataset is validated to ensure that the expected changes were successful and that no unintended data loss or structural changes occurred.

The validation checks confirm:

* The unnecessary source index column has been removed.
* The `Date` column has been converted to `datetime`.
* The dataset adopts a consistent snake_case naming convention.
* The dataset was sorted by date and transaction_id
* The number of records remains unchanged.
* No new missing values have been introduced.
* The transaction-line-item grain has been preserved.


In [18]:
df.columns

Index(['date', 'customer_id', 'transaction_id', 'sku_category', 'sku',
       'quantity', 'sales_amount'],
      dtype='object')

In [19]:
df.dtypes

date              datetime64[ns]
customer_id                int64
transaction_id             int64
sku_category              object
sku                       object
quantity                 float64
sales_amount             float64
dtype: object

In [20]:
df.shape[0]

131706

In [21]:
df.isnull().sum()

date              0
customer_id       0
transaction_id    0
sku_category      0
sku               0
quantity          0
sales_amount      0
dtype: int64

In [22]:
df.isnull().sum().sum()

0

In [23]:
df.groupby('transaction_id').size().value_counts().sort_index()

1     32906
2     16845
3      6802
4      3478
5      1851
6      1036
7       646
8       426
9       254
10      167
11       94
12       64
13       52
14       23
15       16
16        4
17        6
18        4
19        5
20        1
21        2
Name: count, dtype: int64

In [24]:
df[df['transaction_id']==76]

,date,customer_id,transaction_id,sku_category,sku,quantity,sales_amount
125,2016-01-02,4122,76,SJS,4BPPU,0.67,118.35
126,2016-01-02,4122,76,29A,LPLTZ,1.00,3.90
127,2016-01-02,4122,76,N8U,TL4WZ,1.00,4.00
128,2016-01-02,4122,76,9ZX,PV9XZ,4.00,33.60
129,2016-01-02,4122,76,OXH,MM77N,2.00,5.37
130,2016-01-02,4122,76,SJS,U4XSE,2.00,11.02
131,2016-01-02,4122,76,FU5,NOKHJ,2.00,22.26
132,2016-01-02,4122,76,LPF,9F8O1,2.00,12.36
133,2016-01-02,4122,76,IEV,NZQS5,2.00,12.27
134,2016-01-02,4122,76,LPF,GYQNP,2.00,14.40


**findings :**
All ETL transformations were successfully validated. The cleaned dataset contains 131,706 rows and 7 business columns, with no missing values introduced during the transformation process. Column names have been standardized using the snake_case naming convention, the date column has been converted to datetime, and the original transaction-line-item grain has been preserved. The dataset is now ready for loading and downstream dimensional modeling.

# **4. Load**


The final stage of the ETL pipeline stores the validated analytical dataset in the project's processed data directory. Keeping the processed dataset separate from the raw source data preserves the original data while providing a consistent input for downstream modeling and analysis.

In [25]:
assert df.shape[0] == 131706
assert df.isnull().sum().sum() == 0

In [26]:
from pathlib import Path

output_path = Path("../data/processed/scanner_data_clean.csv")


In [27]:
df.to_csv(output_path, index=False)

In [28]:
output_path.exists()

True

**Finding:**


The cleaned dataset was successfully saved to the `data/processed` directory as `scanner_data_clean.csv`. Validation confirmed that the file was written successfully and can be reloaded for downstream analytical workflows.

# **ETL Summary**

✓ Removed source index

✓ Converted dates

✓ Standardized names

✓ Sorted records

✓ Validated integrity

✓ Saved processed dataset

## **ETL Deliverable**

| Output | Description |
|---------|-------------|
| `scanner_data_clean.csv` | Cleaned analytical dataset generated by the ETL pipeline |

The processed dataset will serve as the primary input for the dimensional modeling, SQL analytics, KPI development, dashboard creation, and business analysis phases of the project.

In [ ]:
df['date']


0        2016-01-02
1        2016-01-02
2        2016-01-02
3        2016-01-02
4        2016-01-02
            ...    
131701   2016-12-31
131702   2016-12-31
131703   2016-12-31
131704   2016-12-31
131705   2016-12-31
Name: date, Length: 131706, dtype: datetime64[ns]

: 